# EU_10-12 – Practical Task: Autonomous Logistics with iRobot Create 3
> **Robotics Frameworks (RoF) · FAU Erlangen-Nürnberg · Winter Semester 2025/26**

---

## 📖 System Overview
This notebook documents the logistics task executed on the real iRobot Create 3.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests
yolo = YOLO('yolov8s.pt')
print('Model loaded')

In [ ]:
def simulate_yolo_detection(image_url, target_classes=None, conf_threshold=0.4):
    resp = requests.get(image_url, timeout=10)
    frame = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
    results = yolo(frame, verbose=False)
    for result in results:
        for box in result.boxes:
            if box.conf[0] > conf_threshold:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
                cls_name = result.names[int(box.cls[0])]
                is_target = target_classes and cls_name in target_classes
                colour = (0, 255, 0) if is_target else (128, 128, 128)
                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2 if is_target else 1)
                cv2.putText(frame, f'{cls_name} {box.conf[0]:.2f}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1)
    return frame

frame = simulate_yolo_detection('https://ultralytics.com/images/zidane.jpg', target_classes=['bottle'])
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

In [ ]:
import time, math
class SimRobot:
    def __init__(self): self.x, self.y, self.path = 0.0, 0.0, [(0.0, 0.0)]
    def navigate_to(self, gx, gy):
        print(f'Navigating to ({gx}, {gy})')
        self.x, self.y = gx, gy; self.path.append((gx, gy))
        return 'arrived'

robot = SimRobot()
robot.navigate_to(1.5, 0.5)
robot.navigate_to(0.0, 0.0)
xs, ys = zip(*robot.path)
plt.plot(xs, ys, 'bo-'); plt.title('Robot Path'); plt.show()